# Echem Practice 1

Hands-on with graphene in [Quantum ESPRESSO](https://www.quantum-espresso.org/): structure preparation, SCF/NSCF calculations, density of states, and STM images.

> The notebook assumes the shared folder `/mnt/hgfs/vm_shared/Practice_1` and QE installed in `/home/md/Software/qe-7.5/bin`. Adjust the paths in the setup cell below if your installation differs.

## 1.1 Structure preparation with ASE

In [49]:
# --- Imports and global paths ---------------------------------------------
import glob
import os
import shutil
import subprocess
import xml.etree.ElementTree as ET

import ase
import matplotlib.pyplot as plt
import numpy as np
from ase.io.cube import read_cube_data, write_cube
from ase.visualize import view

# Project folders
BASE_FOLDER = "/mnt/hgfs/vm_shared/Practice_1"
PSEUDO_DIR = "/mnt/hgfs/vm_shared/pseudo"
QE_BIN = "/home/md/Software/qe-7.5/bin"

In [ ]:
# Graphene: hexagonal lattice with a = 2.46 Å (ideally relaxed beforehand)
a = 2.46

cell = np.array([
    [a, 0.0, 0.0],
    [a / 2, a * np.sqrt(3) / 2, 0.0],
    [0.0, 0.0, 20.0],          # 20 Å of vacuum along z isolates the layer
])

# Two carbon atoms of the primitive cell, placed in the middle of the slab
positions = [
    [0.0, 0.0, 10.0],
    [a / 2, a / (2 * np.sqrt(3)), 10.0],
]

graphene = ase.Atoms(
    symbols=["C", "C"],
    positions=positions,
    cell=cell,
)

# 3x3 supercell used in all calculations below
system = graphene.repeat((3, 3, 1))

In [ ]:
# Inspect the generated structure
print("Cell\n")
print(system.get_cell())
print("\nAtom types\n")
print(system.get_chemical_symbols())
print("\nPositions\n")
print(system.get_positions())

In [ ]:
# Optional: open the structure in the ASE GUI (requires a display)
view(system)

### Writing the pw.x input

`write_qe_input` renders a minimal QE input for this carbon-only systems.

In [ ]:
def write_qe_input(filename, atoms, kmesh, calculation="scf",
                   disk_io="nowf", prefix="gr"):
    """Write a minimal Quantum ESPRESSO pw.x input for a carbon structure."""
    kx, ky, kz = kmesh

    # One line per atom: symbol and Cartesian coordinates (Å)
    pos_block = "\n".join(
        f"{symbol:2s} {pos[0]:14.10f} {pos[1]:14.10f} {pos[2]:14.10f}"
        for symbol, pos in zip(atoms.get_chemical_symbols(), atoms.get_positions())
    )

    # One line per lattice vector (Å)
    cell_block = "\n".join(
        f"{v[0]:14.10f} {v[1]:14.10f} {v[2]:14.10f}"
        for v in atoms.get_cell()
    )

    template = f"""&control
  calculation = '{calculation}'
  disk_io = '{disk_io}'
  prefix = '{prefix}'
  pseudo_dir = '{PSEUDO_DIR}'
/
&system
  ibrav = 0
  nat = {len(atoms)}
  ntyp = 1
  nspin = 1
  ecutwfc = 40
  ecutrho = 330
  occupations = 'smearing'
  smearing = 'gaussian'
  degauss = 1e-2
/
&electrons
  conv_thr = 1e-5
  mixing_beta = 0.4
  startingpot = "atomic"
  startingwfc = "atomic+random"
/
ATOMIC_SPECIES
C  12.01070  C.pbe-n-kjpaw_psl.1.0.0.UPF

ATOMIC_POSITIONS {{angstrom}}
{pos_block}

CELL_PARAMETERS {{angstrom}}
{cell_block}

K_POINTS automatic
 {kx} {ky} {kz} 0 0 0
"""
    # Create the destination folder if it does not exist yet
    folder = os.path.dirname(filename)
    if folder:
        os.makedirs(folder, exist_ok=True)

    with open(filename, "w") as f:
        f.write(template)

In [ ]:
# SCF calculation on a 5x5x1 k-mesh, enough for the charge density
write_qe_input("gr/scf_k5/gr_scf.in", system, kmesh=(5, 5, 1), calculation="scf")

## 1.2 Running QE from Python

`subprocess` is enough to drive QE: `pw.x` is launched through `mpirun`, and its log is redirected to the `.out` file.

In [ ]:
def run_qe(inp, out, nproc=2, pw_exe=f"{QE_BIN}/pw.x", cwd=None):
    """Run pw.x with MPI and write its log to `out`."""
    with open(out, "w") as fout:
        subprocess.run(
            ["mpirun", "-np", str(nproc), pw_exe, "-in", inp],
            stdout=fout,
            stderr=subprocess.STDOUT,
            check=True,
            cwd=cwd,
        )

In [ ]:
# Run SCF; `cwd` keeps the QE scratch files next to the input
scf_dir = f"{BASE_FOLDER}/gr/scf_k5"
run_qe(f"{scf_dir}/gr_scf.in", f"{scf_dir}/gr_scf.out", nproc=2, cwd="gr/scf_k5")

## 1.3 Density of states (DOS)

`dos.x` reads the eigenvalues stored in `<prefix>.save` and writes a broadened total DOS to `gr.dos`. Energies in that file are absolute; we shift them by `E_Fermi` when plotting.

In [ ]:
def write_dos_input(filename, prefix="gr", outdir="../",
                    emin=-25, emax=5, deltae=0.01, degauss=0.01):
    """Write a dos.x input (energy grid in eV, relative to E_Fermi)."""
    text = f"""&dos
  outdir = '{outdir}'
  prefix = '{prefix}'
  degauss = {degauss}
  deltae = {deltae}
  emax = {emax}
  emin = {emin}
/
"""
    with open(filename, "w") as f:
        f.write(text)


def run_dos(inp, out, dos_exe=f"{QE_BIN}/dos.x", cwd=None):
    """Run dos.x and write its log to `out`."""
    with open(out, "w") as fout:
        subprocess.run(
            [dos_exe, "-in", inp],
            stdout=fout, stderr=subprocess.STDOUT, check=True, cwd=cwd,
        )

In [ ]:
# `outdir` in dos.in is relative to the working directory, i.e. ../gr.save
dos_dir = f"{BASE_FOLDER}/gr/scf_k5/dos"
os.makedirs(dos_dir, exist_ok=True)

write_dos_input(f"{dos_dir}/dos.in", prefix="gr", outdir="../")
run_dos(f"{dos_dir}/dos.in", f"{dos_dir}/dos.out", cwd=dos_dir)

### Plotting the DOS

The first line of `gr.dos` contains `EFermi`, the rest is a two-column table (energy, total DOS).

In [ ]:
def read_dos(filename):
    """Read energies (eV), total DOS, and E_Fermi from a QE .dos file."""
    with open(filename) as f:
        header = f.readline()
    efermi = float(header.split("EFermi")[1].split("=")[1].split()[0])

    data = np.loadtxt(filename, comments="#")
    energies = data[:, 0]
    dos = data[:, 1]
    return energies, dos, efermi


# DOS of the SCF run, with E_Fermi taken as zero
e, dos, efermi = read_dos("gr/scf_k5/dos/gr.dos")

plt.figure(dpi=100)
plt.plot(e - efermi, dos, label="5x5x1 - scf")
plt.xlabel("E - E_Fermi (eV)")
plt.ylabel("DOS (states/eV)")
plt.legend(frameon=False)
plt.show()

### k-mesh convergence of the DOS

A smooth DOS needs a much denser k-mesh than the SCF run. The standard trick is to reuse the converged charge density (`gr.save`) and to run a non-self-consistent (NSCF) calculation with more k-points.

In [ ]:
for k in [7, 9, 11]:
    run_dir = f"{BASE_FOLDER}/gr/nscf_k{k}"
    
    # Start from the converged SCF density
    if os.path.exists(f"{run_dir}/gr.save"):
        shutil.rmtree(f"{run_dir}/gr.save")
    shutil.copytree("gr/scf_k5/gr.save", f"{run_dir}/gr.save")
    
    # NSCF calculation on a denser k-mesh
    write_qe_input(f"{run_dir}/nscf_k{k}.in", system, (k, k, 1), calculation="nscf")
    run_qe(f"{run_dir}/nscf_k{k}.in", f"{run_dir}/nscf_k{k}.out", nproc=2, cwd=run_dir)
    
    # DOS from the NSCF eigenvalues (dos.in keeps the default outdir = ../)
    dos_dir = f"{run_dir}/dos"
    os.makedirs(dos_dir, exist_ok=True)
    write_dos_input(f"{dos_dir}/dos.in")
    run_dos(f"{dos_dir}/dos.in", f"{dos_dir}/dos.out", cwd=dos_dir)

In [ ]:
plt.figure(dpi=100)
for k in [7, 9, 11]:
    e, dos, efermi = read_dos(f"gr/nscf_k{k}/dos/gr.dos")
    plt.plot(e - efermi, dos, label=f"{k}x{k}x1")

plt.xlabel("E - E_Fermi (eV)")
plt.ylabel("DOS (states/eV)")
plt.legend(frameon=False)
plt.show()

print(f"E_Fermi for {k}x{k}x1 k-mesh: {efermi:.3f} eV")

## 1.4 STM image with QE pp.x

`pp.x` with `plot_num = 10` integrates the LDOS from `emin` up to `E_Fermi` (Tersoff-Hamann model) and writes it as a 3D grid. Wavefunctions have to be kept on disk, so the NSCF step is repeated with `disk_io = 'medium'`.

In [ ]:
# Repeat the NSCF with wavefunctions on disk: pp.x needs them for plot_num = 10
k = 15
run_dir = f"{BASE_FOLDER}/gr/nscf_k{k}"

if os.path.exists(f"{run_dir}/gr.save"):
    shutil.rmtree(f"{run_dir}/gr.save")
shutil.copytree("gr/scf_k5/gr.save", f"{run_dir}/gr.save")

write_qe_input(f"{run_dir}/nscf_k{k}.in", system, (k, k, 1),
               calculation="nscf", disk_io="medium")
run_qe(f"{run_dir}/nscf_k{k}.in", f"{run_dir}/nscf_k{k}.out", nproc=2, cwd=run_dir)

In [ ]:
def write_pp_stm_input(filename, outdir, emin, prefix="gr", fileout="stm.cube"):
    """Write a pp.x input for an STM-like LDOS map.

    plot_num      = 10  integrated LDOS (Tersoff-Hamann)
    emin          lower edge of the energy window, [emin, E_Fermi]
    iflag         = 3   3D data on a regular grid
    output_format = 6   Gaussian cube
    """
    text = f"""&inputpp
  outdir = '{outdir}'
  prefix = '{prefix}'
  plot_num = 10
  emin = {emin}
/
&plot
  iflag = 3
  output_format = 6
  fileout = '{fileout}'
/
"""
    with open(filename, "w") as f:
        f.write(text)


def run_pp(inp, out, cwd=None, pp_exe=f"{QE_BIN}/pp.x"):
    """Run pp.x from its working directory."""
    with open(out, "w") as fout:
        subprocess.run(
            [pp_exe, "-in", os.path.basename(inp)],
            stdout=fout, stderr=subprocess.STDOUT, check=True, cwd=cwd,
        )

In [ ]:
os.makedirs("gr/stm", exist_ok=True)

# ! `efermi` comes from the DOS plot above
write_pp_stm_input(
    "gr/stm/stm.in",
    outdir=f"{BASE_FOLDER}/gr/nscf_k15",
    emin=efermi - 1.0,      # bias of 1 eV below E_Fermi
    fileout="stm_1eV.cube",
)
run_pp("gr/stm/stm.in", "gr/stm/stm.out", cwd="gr/stm")

In [ ]:
def _slice(cube_path, z_position=2.0):
    """Cut a constant-height plane `z_position` Å above the topmost atom."""
    data, atoms = read_cube_data(cube_path)
    nx, ny, nz = data.shape

    z_height = atoms.positions[:, 2].max() + z_position
    z_index = int(z_height / atoms.cell[2, 2] * nz)
    slice_2d = data[:, :, z_index]

    # Grid indices -> Cartesian coordinates in the xy-plane
    a1, a2 = atoms.cell[0], atoms.cell[1]
    i, j = np.meshgrid(np.arange(nx), np.arange(ny), indexing="ij")
    X = (i / nx) * a1[0] + (j / ny) * a2[0]
    Y = (i / nx) * a1[1] + (j / ny) * a2[1]

    z = atoms.cell[2, 2] * z_index / nz
    return X, Y, slice_2d, z


def show_slice(cube_path, z_position=2.0, cmap="inferno"):
    """Plot one constant-height LDOS slice as a heat map."""
    X, Y, slice_2d, z = _slice(cube_path, z_position=z_position)

    plt.figure(figsize=(7, 6))
    plt.pcolormesh(X, Y, slice_2d, cmap=cmap)
    plt.colorbar(label=r"$|\psi|^2$", shrink=0.5, pad=0.04)
    plt.axis("scaled")
    plt.xlabel("x (Å)")
    plt.ylabel("y (Å)")
    plt.title(f"LDOS @ z = {z:.2f} Å")
    plt.tight_layout()
    plt.show()

In [ ]:
# STM image produced by pp.x
show_slice("gr/stm/stm_1eV.cube", z_position=2)

## 1.5 STM image from scratch

The same image can be assembled manually:

1. read eigenvalues and occupations from `data-file-schema.xml`;
2. select the states inside the energy window;
3. dump `|psi|^2` of every state with `pp.x` (`plot_num = 7`);
4. sum the cubes with the weights `occupation x k-point weight`.

This is exactly what an STM simulation does under the hood.

In [ ]:
HARTREE_TO_EV = 27.211386245988


def read_states(xml_path):
    """Read E_Fermi, eigenvalues, k-point weights, and occupations from QE XML."""
    root = ET.parse(xml_path).getroot()

    # Drop XML namespaces to keep the tag names simple
    for el in root.iter():
        if "}" in el.tag:
            el.tag = el.tag.split("}", 1)[1]

    e_fermi = float(root.find(".//fermi_energy").text)
    bs = root.find(".//band_structure")
    nbnd = int(bs.find("nbnd").text)
    nks = int(bs.find("nks").text)

    eig = np.zeros((nks, nbnd))
    occs = np.zeros((nks, nbnd))
    weights = np.zeros(nks)
    kpts = np.zeros((nks, 3))

    for i, ks in enumerate(bs.findall("ks_energies")):
        kp = ks.find("k_point")
        kpts[i] = np.array(kp.text.split(), dtype=float)[:3]
        weights[i] = float(kp.get("weight"))
        eig[i] = np.array(ks.find("eigenvalues").text.split(), dtype=float)
        occs[i] = np.array(ks.find("occupations").text.split(), dtype=float)

    return e_fermi, eig, weights, occs, kpts


def states_in_window(eig, weights, occs, e_fermi, emin_eV, emax_eV):
    """List all (k-point, band) states inside [emin_eV, emax_eV] around E_Fermi."""
    # Convert the window from eV to Hartree (QE stores energies in Hartree)
    emin = e_fermi + emin_eV / HARTREE_TO_EV
    emax = e_fermi + emax_eV / HARTREE_TO_EV

    nks, nbnd = eig.shape
    states = []
    for ik in range(nks):
        for ib in range(nbnd):
            if emin <= eig[ik, ib] <= emax:
                states.append({
                    "ik": ik + 1,      # QE numbers k-points and bands from 1
                    "ib": ib + 1,
                    "energy": eig[ik, ib],
                    "weight": weights[ik],
                    "occ": occs[ik, ib],
                })
    return states


def run_pp_all(outdir_qe, prefix, states, workdir, pp_exe=f"{QE_BIN}/pp.x"):
    """Dump |psi|^2 for every selected state with pp.x (plot_num = 7)."""
    os.makedirs(workdir, exist_ok=True)

    for state in states:
        ik, ib = state["ik"], state["ib"]
        cube = os.path.join(workdir, f"psi2_k{ik}_b{ib}.cube")
        inp = os.path.join(workdir, f"pp_k{ik}_b{ib}.in")
        out = os.path.join(workdir, f"pp_k{ik}_b{ib}.out")

        if os.path.exists(cube):
            print(f"skip k={ik} b={ib}")
            continue

        with open(inp, "w") as f:
            f.write(f"""&inputpp
  outdir = '{outdir_qe}'
  prefix = '{prefix}'
  plot_num = 7
  kpoint = {ik}
  kband = {ib}
/
&plot
  iflag = 3
  output_format = 6
  fileout = '{cube}'
/
""")

        with open(out, "w") as log:
            subprocess.run([pp_exe, "-in", inp],
                           stdout=log, stderr=subprocess.STDOUT, check=True)

        print(f"Done for k={ik} b={ib} -> {cube}")

In [ ]:
e_fermi, eig, weights, occs, kpts = read_states("gr/nscf_k15/gr.save/data-file-schema.xml")

# Energy window relative to E_Fermi (eV)
emin_eV, emax_eV = -1.0, 0.0
states = states_in_window(eig, weights, occs, e_fermi, emin_eV, emax_eV)

print(f"Number of states in the window from {emin_eV} to {emax_eV} eV: {len(states)}")

In [ ]:
# Dump |psi|^2 for each selected state; existing cubes are skipped
workdir = "gr/nscf_k15/pp"
run_pp_all("gr/nscf_k15", "gr", states, workdir)

In [ ]:
def show_all_slices(folder, pattern="*.cube", z_position=3, ncols=4,
                    cmap="inferno"):
    """Plot constant-height slices for all cube files found in `folder`."""
    paths = sorted(glob.glob(os.path.join(folder, pattern)))
    nrows = -(-len(paths) // ncols)          # ceil division

    fig, axes = plt.subplots(nrows, ncols, figsize=(3.2 * ncols, 3 * nrows),
                             squeeze=False, constrained_layout=True)

    for ax, path in zip(axes.ravel(), paths):
        X, Y, slice_2d, z = _slice(path, z_position)
        pc = ax.pcolormesh(X, Y, slice_2d, cmap=cmap, shading="auto")
        ax.set_aspect("equal")
        ax.tick_params(labelsize=7, length=2)
        ax.set_title(f"{os.path.basename(path)}\nz = {z:.2f} Å", fontsize=7.5)
        fig.colorbar(pc, ax=ax, shrink=0.5, pad=0.05)

    plt.show()

In [ ]:
# Individual states contributing to the STM image
show_all_slices("gr/nscf_k15/pp", z_position=2)

In [ ]:
def sum_cubes(states, filepath, workdir="."):
    """Sum |psi|^2 over states weighted by occupation x k-point weight."""
    data_sum, atoms = None, None

    for state in states:
        cube = os.path.join(workdir, f"psi2_k{state['ik']}_b{state['ib']}.cube")
        data, atoms = read_cube_data(cube)
        weight = state["occ"] * state["weight"]
        data_sum = data * weight if data_sum is None else data_sum + data * weight

    os.makedirs(os.path.dirname(filepath), exist_ok=True)
    with open(filepath, "w") as f:
        write_cube(f, atoms, data=data_sum,
                   comment=f"LDOS sum over {len(states)} states")
    print(f"saved: {filepath}")

In [ ]:
sum_cubes(states, "gr/nscf_k15/STM_self/rho_1eV.cube", workdir="gr/nscf_k15/pp")

In [ ]:
# Compare: pp.x STM vs. the weighted sum of |psi|^2
show_slice("gr/stm/stm_1eV.cube", z_position=2)
show_slice("gr/nscf_k15/STM_self/rho_1eV.cube", z_position=2)

# Homework 1

1. Explain the difference between the STM image produced by QE and the self-made version. *Hint:* check the k-mesh in the output file. Is this the k-mesh you expected?
2. Propose a solution (or several solutions) that would give an STM image identical to the QE one.
3. Calculate the STM image of graphene with a monovacancy and compare it with experimental data from papers.
4. \* Calculate STM images beyond the Tersoff-Hamann approximation (Chen's) and compare them with the literature.

You may use LLMs while preparing the homework. Just declare the models used and their purpose, and remember that you are responsible for the validity of the results.

- **Deadline:** November 10, 2026, 23:59 UTC
- **Format:** one PDF file with the solution
- **Submission:** sergey.v.pavlov@phystech.edu or Telegram @sergey_v_pavlov

**Declaration of Generative AI:** LLM models (deepseek-v4.1-flash and nemotron-3-ultra) were used during preparation of this tutorial for coding, code polishing, adding comments and markdown cells. After using these tools, the author reviewed, verified, and edited the content as needed, and take full responsibility for the final version.